In [4]:
import numpy as np
from decisionTree import DecisionTree

In [ ]:
class RandomForest:
    def __init__(self, n_trees=10, max_depth=None, max_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.max_features = max_features  # e.g. sqrt(n_features) — used inside each tree's split search
        self.trees = []
 
    def _bootstrap_sample(self, X, y):
        #Draw a random sample of rows, same size as X, with replacement.
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        return X[indices], y[indices]
 
    def fit(self, X, y):
        self.trees = []
 
        for _ in range(self.n_trees):
            X_sample, y_sample = self._bootstrap_sample(X, y)
 
            tree = DecisionTree(max_depth=self.max_depth, max_features=self.max_features)
            tree.fit(X_sample, y_sample)
 
            self.trees.append(tree)
 
        return self
 
    def predict(self, X):
        # each row: predictions from every tree for that sample
        all_predictions = np.array([tree.predict(X) for tree in self.trees])  # shape (n_trees, n_samples)
 
        # transpose so each row is one sample's predictions across all trees
        all_predictions = all_predictions.T  # shape (n_samples, n_trees)
 
        majority_votes = np.array([
            np.bincount(sample_preds.astype(int)).argmax()  #majority vote for each sample
            for sample_preds in all_predictions
        ])
 
        return majority_votes
 
    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y)

In [8]:

if __name__ == "__main__":
    from sklearn.datasets import load_iris
    from sklearn.model_selection import train_test_split
 
    X,y = load_iris(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
 
    forest = RandomForest(n_trees=10, max_depth=3, max_features=2)
    forest.fit(X_train, y_train)
 
    train_acc = forest.score(X_train, y_train)
    test_acc = forest.score(X_test, y_test)
 
    print(f"Random Forest — Train accuracy: {train_acc:.4f}")
    print(f"Random Forest — Test accuracy:  {test_acc:.4f}")
 
    # compare against a single tree with the same max_depth
    single_tree = DecisionTree(max_depth=3)
    single_tree.fit(X_train, y_train)
    print(f"\nSingle Tree  — Train accuracy: {single_tree.score(X_train, y_train):.4f}")
    print(f"Single Tree  — Test accuracy:  {single_tree.score(X_test, y_test):.4f}")

Random Forest — Train accuracy: 0.9500
Random Forest — Test accuracy:  1.0000

Single Tree  — Train accuracy: 0.9583
Single Tree  — Test accuracy:  1.0000


In [ ]:
#4.7
#comparing single tree with random forest on a noisy dataset

dt = DecisionTree(max_depth=3)
rf = RandomForest(n_trees=10, max_depth=3, max_features=2)

from sklearn.datasets import make_classification
def load_noisy_dataset(n_samples=300, flip_y=0.10, random_state=42, test_size=0.2):
    """Synthetic noisy dataset for demonstrating overfitting (Item 4.3)."""
    X, y = make_classification(
        n_samples=n_samples,
        n_features=10,
        n_informative=5,
        n_redundant=2,
        n_classes=2,
        flip_y=flip_y,
        random_state=random_state,
    )
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

X_train, X_test, y_train, y_test = load_noisy_dataset(flip_y=0.10)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

print(f"Single Tree  — Train accuracy: {dt.score(X_train, y_train):.4f}")
print(f"Single Tree  — Test accuracy:  {dt.score(X_test, y_test):.4f}")
print(f"\nRandom Forest — Train accuracy: {rf.score(X_train, y_train):.4f}")
print(f"Random Forest — Test accuracy:  {rf.score(X_test, y_test):.4f}")

#testing both on overfitting

dt = DecisionTree(max_depth=None)
rf = RandomForest(n_trees=10, max_depth=None, max_features=2)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

print("\n Overfitting results: \n")
print(f"Single Tree  — Train accuracy: {dt.score(X_train, y_train):.4f}")
print(f"Single Tree  — Test accuracy:  {dt.score(X_test, y_test):.4f}")
print(f"\nRandom Forest — Train accuracy: {rf.score(X_train, y_train):.4f}")
print(f"Random Forest — Test accuracy:  {rf.score(X_test, y_test):.4f}")

print("We can see that in both the cases, the random forest performs better than a single decision tree, especially on the test set, indicating that it is less prone to overfitting.")


Single Tree  — Train accuracy: 0.8708
Single Tree  — Test accuracy:  0.6667

Random Forest — Train accuracy: 0.8500
Random Forest — Test accuracy:  0.6833

 Overfitting results: 

Single Tree  — Train accuracy: 1.0000
Single Tree  — Test accuracy:  0.6333

Random Forest — Train accuracy: 0.9917
Random Forest — Test accuracy:  0.7500
